# Appendix 10. scikit-learn 기초

## 1. Goal

이 notebook은 scikit-learn의 공통 estimator API와 tabular classification workflow를 익히는 자료입니다. `X`와 `y`, train/validation split, transformer와 predictor, `ColumnTransformer`, `Pipeline`, metric, cross-validation과 parameter search를 하나의 흐름으로 연결합니다.

점수를 높이는 것이 아니라 preprocessing을 train data에만 fit하고, 동일한 Pipeline을 validation data에 적용해 leakage를 막는 구조를 이해하는 것이 목표입니다.

## 2. Setup

프로젝트에 설치된 scikit-learn과 pandas, NumPy를 사용합니다. 실제 PhysioNet data나 sealed test는 읽지 않고 고정 seed로 만든 합성 데이터만 사용합니다.

In [ ]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

print({"scikit-learn": sklearn.__version__, "pandas": pd.__version__})


## 3. Steps

### 3-1. 데이터와 estimator API

#### 3-1-1. feature matrix X와 target y

scikit-learn의 supervised estimator는 보통 `(n_samples, n_features)` 형태의 `X`와 각 sample에 대응하는 1차원 `y`를 받습니다. pandas DataFrame을 사용하면 column name과 dtype을 유지할 수 있습니다. 아래 데이터는 API 실습용일 뿐 임상 규칙이나 실제 risk model을 나타내지 않습니다.

In [ ]:
rng = np.random.default_rng(42)
sample_count = 120
feature_frame = pd.DataFrame(
    {
        "age": rng.integers(18, 90, size=sample_count),
        "heart_rate": rng.normal(86, 16, size=sample_count),
        "lactate": rng.lognormal(0.6, 0.45, size=sample_count),
        "unit": rng.choice(["ICU-A", "ICU-B"], size=sample_count),
    }
)
risk_signal = (
    0.04 * (feature_frame["age"] - 60)
    + 0.05 * (feature_frame["heart_rate"] - 86)
    + 0.8 * (feature_frame["lactate"] - 2.0)
    + 0.4 * feature_frame["unit"].eq("ICU-B")
    + rng.normal(0, 0.8, size=sample_count)
)
target = risk_signal.gt(risk_signal.median()).astype("int8").rename("target")

feature_frame.loc[[3, 17, 41], "lactate"] = np.nan
feature_frame.loc[[8, 29], "unit"] = None
print({"X_shape": feature_frame.shape, "y_shape": target.shape})
feature_frame.head()


#### 3-1-2. train과 validation 분리

split은 preprocessing보다 먼저 수행합니다. `stratify=y`는 각 split의 class 비율을 비슷하게 유지하고, `random_state`는 같은 입력에서 같은 split을 재현합니다. 실제 프로젝트에서는 비율과 seed를 versioned config에서 가져와야 합니다.

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    feature_frame,
    target,
    test_size=0.25,
    random_state=42,
    stratify=target,
)
split_summary = pd.DataFrame(
    {
        "row_count": [len(X_train), len(X_valid)],
        "positive_rate": [y_train.mean(), y_valid.mean()],
    },
    index=["train", "valid"],
)
split_summary


#### 3-1-3. fit, predict와 predict_proba

estimator는 constructor에서 hyperparameter를 받고 `fit()`에서 data를 학습합니다. fitted estimator는 `predict()`로 class를, 확률을 지원하는 classifier는 `predict_proba()`로 class별 probability를 반환합니다. 먼저 feature를 사용하지 않는 `DummyClassifier`로 비교 기준을 만듭니다.

In [ ]:
dummy = DummyClassifier(strategy="prior")
dummy.fit(X_train, y_train)
dummy_probabilities = dummy.predict_proba(X_valid)[:, 1]
dummy_summary = {
    "classes": dummy.classes_.tolist(),
    "positive_probability": float(dummy_probabilities[0]),
    "roc_auc": roc_auc_score(y_valid, dummy_probabilities),
}
dummy_summary


### 3-2. preprocessing과 Pipeline

#### 3-2-1. numeric과 categorical column 변환

transformer는 `fit()`에서 train data의 통계나 category를 학습하고 `transform()`으로 새 feature matrix를 만듭니다. `ColumnTransformer`는 column별로 다른 변환을 적용합니다. numeric column에는 median imputation과 scaling을, categorical column에는 most-frequent imputation과 one-hot encoding을 적용합니다.

In [ ]:
numeric_features = ["age", "heart_rate", "lactate"]
categorical_features = ["unit"]

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features),
    ]
)
preprocessor


#### 3-2-2. Pipeline로 변환과 model 연결

`Pipeline`은 여러 transformer와 마지막 estimator를 하나의 estimator처럼 다루게 합니다. validation data에는 train data로 fitted된 preprocessing만 적용되므로 직접 `fit_transform()`을 반복할 때 생기기 쉬운 leakage를 줄입니다. step은 `named_steps`로, 중첩 parameter는 `step__parameter` 이름으로 접근합니다.

In [ ]:
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)
pipeline.set_params(model__C=1.0)
pipeline.fit(X_train, y_train)

valid_probabilities = pipeline.predict_proba(X_valid)[:, 1]
valid_predictions = pipeline.predict(X_valid)
feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()
print(
    {
        "feature_count": len(feature_names),
        "feature_names": feature_names.tolist(),
        "model_C": pipeline.get_params()["model__C"],
    }
)


### 3-3. 평가와 model selection

#### 3-3-1. probability metric과 confusion matrix

ROC AUC와 Average Precision은 probability ranking을 평가하고, confusion matrix는 정한 threshold에서 class decision을 요약합니다. 같은 model이라도 threshold에 따라 confusion matrix가 달라지므로 probability metric과 decision metric을 구분합니다.

In [ ]:
validation_metrics = {
    "roc_auc": roc_auc_score(y_valid, valid_probabilities),
    "average_precision": average_precision_score(
        y_valid, valid_probabilities
    ),
}
validation_confusion = confusion_matrix(y_valid, valid_predictions)
print(validation_metrics)
validation_confusion


#### 3-3-2. cross-validation

한 번의 split 결과는 우연한 sample 구성에 민감할 수 있습니다. `cross_validate()`는 여러 fold에서 동일한 Pipeline을 다시 fit하고 score를 반환합니다. `StratifiedKFold`는 classification fold의 class 비율을 유지하며, shuffle을 사용할 때는 seed를 고정합니다.

In [ ]:
cross_validation = StratifiedKFold(
    n_splits=4, shuffle=True, random_state=42
)
cv_result = cross_validate(
    pipeline,
    X_train,
    y_train,
    cv=cross_validation,
    scoring={"roc_auc": "roc_auc", "average_precision": "average_precision"},
    return_train_score=False,
)
cv_summary = pd.DataFrame(
    {
        "roc_auc": cv_result["test_roc_auc"],
        "average_precision": cv_result["test_average_precision"],
    }
)
cv_summary


#### 3-3-3. Pipeline parameter search

`GridSearchCV`는 각 parameter 조합을 cross-validation으로 비교합니다. preprocessing까지 포함한 Pipeline 전체를 넘겨야 각 fold의 train 부분에서만 imputer와 scaler가 fit됩니다. 실제 tuning에서는 탐색 공간, primary metric과 tie-break rule을 먼저 versioned policy로 정합니다.

In [ ]:
parameter_search = GridSearchCV(
    estimator=pipeline,
    param_grid={"model__C": [0.1, 1.0, 10.0]},
    scoring="roc_auc",
    cv=cross_validation,
    refit=True,
)
parameter_search.fit(X_train, y_train)
search_summary = pd.DataFrame(parameter_search.cv_results_)[
    ["param_model__C", "mean_test_score", "rank_test_score"]
].sort_values("rank_test_score")
print({"best_params": parameter_search.best_params_})
search_summary


## 4. Checks

split, Pipeline, probability와 cross-validation 결과의 shape와 범위를 확인합니다. 이 합성 데이터의 점수는 학습 목표가 아니므로 특정 성능 기준으로 model을 승인하지 않습니다.

In [ ]:
assert feature_frame.shape == (120, 4)
assert len(X_train) + len(X_valid) == 120
assert pipeline.classes_.tolist() == [0, 1]
assert valid_probabilities.shape == (len(X_valid),)
assert np.all((valid_probabilities >= 0) & (valid_probabilities <= 1))
assert validation_confusion.shape == (2, 2)
assert cv_summary.shape == (4, 2)
assert parameter_search.best_params_["model__C"] in {0.1, 1.0, 10.0}
print("All appendix checks passed.")


## 5. Next Steps

- dataset role을 먼저 나누고 preprocessing과 model을 하나의 Pipeline으로 묶습니다.
- `DummyClassifier`를 feature를 사용하는 model의 최소 비교 기준으로 둡니다.
- imbalance가 있는 classification에서는 accuracy 하나로 결론 내리지 않습니다.
- threshold tuning은 probability model selection과 분리하고 train/validation 범위에서 수행합니다.
- fitted Pipeline을 저장할 때 feature contract, library version과 data provenance를 함께 기록합니다.

## 6. References

이 notebook의 설명과 예제는 다음 공식 문서를 기준으로 작성했습니다.

- [Getting Started](https://scikit-learn.org/stable/getting_started.html)
- [Pipelines and composite estimators](https://scikit-learn.org/stable/modules/compose.html)
- [Cross-validation: evaluating estimator performance](https://scikit-learn.org/stable/modules/cross_validation.html)
- [Metrics and scoring](https://scikit-learn.org/stable/modules/model_evaluation.html)
- [Common pitfalls and recommended practices](https://scikit-learn.org/stable/common_pitfalls.html)